# Definição de Funções, Imports e Variáveis

In [0]:
# Definindo Imports e variáveis

catalogo = 'medalhao_rocket'
silver = 'silver'

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, unix_timestamp, to_timestamp, upper, expr
from pyspark.sql import functions as F

In [0]:

# Recebe entrada estruturada da seguinte forma
# DF -> DataFrame Pyspark
# dicionario_nomes -> {nome_antigo:nome_novo}
# table_name -> nome da tabela que será criada após renomear
def rename_columns(df, dicionario_nomes, table_name):
    df_renamed = df.select([
        col(c).alias(dicionario_nomes.get(c, c))
        for c in df.columns
    ])

    df_renamed = df_renamed.withColumn('data_ingestao_silver', F.current_timestamp())
    df_renamed.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.{table_name}")

    return df_renamed

# Tratamento da Tabela 
## tb_customers -> dim_consumidores

### Mudanças
- Nomes das colunas alternados para o português
- Deduplicação id_unico_consumidor
- Nomes de Cidade e Estado em Upper Case

In [0]:
# Renomando as colunas da tabela tb_customers

nomes = {'customer_id': 'id_consumidor', 'customer_zip_code_prefix': 'prefixo_cep', 'customer_name': 'nome_consumidor', 'customer_city': 'cidade', 'customer_state': 'estado', 'customer_age': 'idade', 'customer_unique_id': 'id_unico_consumidor', 'customer_birth_date': 'data_nascimento_consumidor', 'customer_gender': 'genero','timestamp_ingestion':'tempo_ingestao_bronze'}

df_customers_bronze = spark.read.table(f"{catalogo}.bronze.tb_customers")

df_customers_silver = rename_columns(df_customers_bronze, nomes, 'dim_consumidores')

In [0]:
# Removendo duplicatas com base na senioridade (timestamp_ingestion mais antigo)

# Definindo o parâmetro que será utilizado para deduplicação
window = Window.partitionBy('id_unico_consumidor').orderBy(col('tempo_ingestao_bronze').desc())
df_customers_silver = spark.read.table(f"{catalogo}.{silver}.dim_consumidores")

# Filtragem dos dados com base no parâmetro definido anteriormente
df_dedup = (
    df_customers_silver
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f'Pre deduplicação senior: {df_customers_silver.count()}')
print(f'Pós deduplicação senior: {df_dedup.count()}')

# Escrevendo na tabela os dados deduplicados
df_dedup.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.dim_consumidores")

In [0]:
# Padronizando os nomes dos valores das colunas de cidade e estado como upper

df_customers_silver = spark.read.table(f"{catalogo}.{silver}.dim_consumidores")

print(f'Valores distintos em cidades pre-padronização: {df_customers_silver.select("cidade").distinct().count()}\nValores distintos em estados pre-padronização:{df_customers_silver.select("estado").distinct().count()}\n')

df_customers_silver = df_customers_silver.withColumn('cidade', upper('cidade'))
df_customers_silver = df_customers_silver.withColumn('estado', upper('estado'))

print(f'Valores distintos em cidades pre-padronização: {df_customers_silver.select("cidade").distinct().count()}\nValores distintos em estados pre-padronização:{df_customers_silver.select("estado").distinct().count()}')

df_customers_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.dim_consumidores")

# Tratamento da Tabela 
## tb_orders -> fat_pedidos

### Mudanças
- Nomes das colunas alternados para o português
- Deduplicação id_unico_consumidor
- Alteração dos valores contidos na coluna de status, traduzindo para o português

### Adições de colunas
- Coluna de Tempo de Entrega em dias (Diferença entre data de entrega real e data de compra)
- Tempo estimado em dias (Diferença entre data estimada e data compra)
- Diferença da entrega em dias (Diferença entre tempo real e estimado)
- Entrega no prazo (Possui os valores "Sim", "Não" e "Não Entregue", baseado na coluna de status)

In [0]:
df_orders_bronze = spark.read.table(f"{catalogo}.bronze.tb_orders")

nomes = {'order_id':'id_pedido','customer_id':'id_consumidor','order_status':'status_pedido','order_purchase_timestamp':'data_compra','order_approved_at':'data_aprovacao','order_delivered_carrier_date':'data_entrega_transporte','order_delivered_customer_date':'data_entrega_cliente','order_estimated_delivery_date':'data_estimada_entrega', 'timestamp_ingestion':'tempo_ingestao_bronze'}

df_orders_silver = rename_columns(df_orders_bronze, nomes, 'fat_pedidos')

In [0]:
valores_substituicao = {'delivered':'entregue', 'canceled':'cancelado', 'shipped':'enviado','processing':'processando','invoiced':'faturado','unavailable':'indisponivel','created':'criado','approved':'aprovado', 'timestamp_ingestion':'tempo_ingestao_bronze'}

df_orders_bronze = spark.read.table(f'{catalogo}.bronze.tb_orders')
df_orders_silver = spark.read.table(f"{catalogo}.silver.fat_pedidos")

df_orders_silver = df_orders_silver.replace(valores_substituicao, subset=['status_pedido'])

df_orders_bronze.select('order_status').distinct().show()
df_orders_silver.select('status_pedido').distinct().show()

df_orders_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.fat_pedidos")

In [0]:
# Criação das novas colunas de tempo_entrega_dias, tempo_entrega_estimado_dias, diferenca_entrega_dias, entrega_no_prazo
df_orders_silver = spark.read.table('medalhao_rocket.silver.fat_pedidos')

df_orders_silver = df_orders_silver.withColumn('tempo_entrega_dias', F.datediff('data_entrega_cliente', 'data_compra'))

df_orders_silver = df_orders_silver.withColumn('tempo_entrega_estimado_dias', F.datediff('data_estimada_entrega', 'data_compra'))

df_orders_silver = df_orders_silver.withColumn('diferenca_entrega_dias', F.datediff('data_estimada_entrega', 'data_entrega_cliente'))

df_orders_silver = df_orders_silver.withColumn(
    'entrega_no_prazo',
    F.when(
        (F.col('status_pedido') == 'entregue') & 
        (F.col('diferenca_entrega_dias') >= 0),
        'Sim'
    )
    .when(
        (F.col('status_pedido') == 'entregue') & 
        (F.col('diferenca_entrega_dias') < 0),
        'Não'
    )
    .otherwise('Não Entregue')
)

df_orders_silver.write.format("delta").mode("overwrite").option('mergeSchema', 'true').saveAsTable(f"{catalogo}.{silver}.fat_pedidos")

# Tratamento da Tabela 
## tb_order_items -> fat_itens_pedidos

### Mudanças
- Nomes das colunas alternados para o português

In [0]:
df_orders_items_bronze = spark.read.table(f"{catalogo}.bronze.tb_order_items")

nomes = {'order_id':'id_pedido','order_item_id':'id_item_pedido','product_id':'id_produto','seller_id':'id_vendedor','shipping_limit_date':'data_limite_entrega','price':'preco','freight_value':'frete', 'timestamp_ingestion':'tempo_ingestao_bronze'}

df_orders_silver = rename_columns(df_orders_items_bronze, nomes, 'fat_itens_pedidos')

# Tratamento da Tabela 
## tb_orders_payment -> fat_pagamentos_pedidos

### Mudanças
- Nomes das colunas alternados para o português
- Valores da coluna tipo_pagamento traduzidos para o português

In [0]:
df_order_payments_bronze = spark.read.table(f"{catalogo}.bronze.tb_order_payments")

nomes = {'order_id':'id_pedido','payment_sequential':'sequencial_pagamento','payment_type':'tipo_pagamento','payment_installments':'parcelas_pagamento','payment_value':'valor_pagamento', 'timestamp_ingestion':'tempo_ingestao_bronze'}

df_orders_silver = rename_columns(df_order_payments_bronze, nomes, 'fat_pagamentos_pedidos')

In [0]:
# Alteração dos valores contidos na coluna de tipo_pagamento
df_order_payments_bronze = spark.read.table(f"{catalogo}.bronze.tb_order_payments")

valores_substituicao = {'credit_card':'Cartão de Crédito', 'boleto':'Boleto', 'voucher':'Voucher','debit_card':'Cartão de Débito','not_defined':'Não Definido'}

df_orders_silver = spark.read.table(f"{catalogo}.silver.fat_pagamentos_pedidos")

df_order_payments_silver = df_orders_silver.replace(valores_substituicao, subset=['tipo_pagamento'])

df_order_payments_bronze.select('payment_type').distinct().show()
df_order_payments_silver.select('tipo_pagamento').distinct().show()

df_order_payments_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.fat_pagamentos_pedidos")

# Tratamento da Tabela 
## tb_order_reviews -> fat_avaliacoes_pedidos

### Mudanças
- Nomes das colunas alternados para o português
- Remoção de registros com pedidos inválidos ou datas de análises no futuro
- Tratamento de nulos nas colunas de titulo e comentario da avaliação
  - Substituindo os nulos em "Sem comentário e Sem título"
- Correção da contaminação de dados ao importar

In [0]:
df_bronze_reviews = spark.read.table(f"{catalogo}.bronze.tb_order_reviews")

names = {'review_id':'id_avaliacao','order_id':'id_pedido','review_score':'nota_avaliacao','review_comment_title':'titulo_avaliacao_comentario','review_comment_message':'mensagem_avaliacao_comentario','review_creation_date':'data_avaliacao','review_answer_timestamp':'data_resposta_avaliacao','timestamp_ingestion':'tempo_ingestao_bronze'}

df_orders_review_silver = rename_columns(df_bronze_reviews, names, 'fat_avaliacoes_pedidos')

In [0]:
from pyspark.sql.functions import when, isnull, col

df_orders_review_silver = spark.read.table('medalhao_rocket.silver.fat_avaliacoes_pedidos')

df_orders_review_silver = df_orders_review_silver.withColumn(
    "titulo_avaliacao_comentario",
    when(isnull(col("titulo_avaliacao_comentario")), "Sem título").otherwise(col("titulo_avaliacao_comentario"))
).withColumn(
    "mensagem_avaliacao_comentario",
    when(isnull(col("mensagem_avaliacao_comentario")), "Sem comentário").otherwise(col("mensagem_avaliacao_comentario"))
)

df_orders_review_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.fat_avaliacoes_pedidos")

In [0]:
# Executando limpeza de ids de pedidos inconsistentes

df_orders_silver = spark.read.table("medalhao_rocket.silver.fat_pedidos")
df_orders_review_silver = spark.read.table("medalhao_rocket.silver.fat_avaliacoes_pedidos")

df_orders_review_silver = df_orders_review_silver.join(df_orders_silver.select('id_pedido'), on="id_pedido", how="left_semi")

df_orders_review_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.fat_avaliacoes_pedidos")

In [0]:
# Limpeza de pedidos com datas futuras
df_orders_review_silver = spark.read.table("medalhao_rocket.silver.fat_avaliacoes_pedidos")

df_orders_review_silver = df_orders_review_silver.filter(
    expr("try_to_timestamp(data_avaliacao)") <
    expr("try_to_timestamp(tempo_ingestao_bronze)")
)

df_orders_review_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.fat_avaliacoes_pedidos")

In [0]:
display(df_orders_review_silver)

# Tratamento da Tabela 
## tb_products -> dim_produtos

### Mudanças
- Nomes das colunas alternados para o português
- Remoção de Duplicatas através da deduplicação senior

In [0]:
df_products_bronze = spark.read.table(f"{catalogo}.bronze.tb_products")

nomes = {'product_id':'id_produto','product_name':'nome_produto','product_category_name':'categoria_produto','product_description_lenght':'tamanho_descricao_produto','product_name_lenght':'tamanho_nome_produto','product_photos_qty':'quantidade_fotos','product_weight_g':'peso_produto_gramas','product_length_cm':'comprimento_centimetros','product_height_cm':'altura_centimetros','product_width_cm':'largura_centimetros','timestamp_ingestion':'tempo_ingestao_bronze'}

df_products_silver = rename_columns(df_products_bronze, nomes, 'dim_produtos')

In [0]:
# Removendo duplicatas com base na senioridade (timestamp_ingestion mais antigo)
window = Window.partitionBy('id_produto').orderBy(col('tempo_ingestao_bronze').desc())
df_products_silver = spark.read.table(f"{catalogo}.{silver}.dim_produtos")

df_dedup = (
    df_products_silver
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f'Pre deduplicação senior: {df_products_silver.count()}')
print(f'Pós deduplicação senior: {df_dedup.count()}')

df_dedup.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.dim_produtos")

# Tratamento da Tabela 
## tb_sellers -> dim_sellers

### Mudanças
- Nomes das colunas alternados para o português
- Remoção de Duplicatas através da deduplicação senior

In [0]:
df_sellers_bronze = spark.read.table(f"{catalogo}.bronze.tb_sellers")

nomes = {'seller_id':'id_vendedor','seller_name':'nome_vendedor','seller_zip_code_prefix':'prefixo_cep','seller_city':'cidade','seller_state':'estado','seller_registration_date':'data_registro_vendedor','timestamp_ingestion':'tempo_ingestao_bronze'}

df_sellers_silver = rename_columns(df_sellers_bronze, nomes, 'dim_vendedores')

In [0]:
df_customers_silver = spark.read.table(f"{catalogo}.{silver}.dim_vendedores")

print(f'Valores distintos em cidades pre-padronização: {df_customers_silver.select("cidade").distinct().count()}\nValores distintos em estados pre-padronização:{df_customers_silver.select("estado").distinct().count()}\n')

df_customers_silver = df_customers_silver.withColumn('cidade', upper('cidade'))
df_customers_silver = df_customers_silver.withColumn('estado', upper('estado'))

print(f'Valores distintos em cidades pre-padronização: {df_customers_silver.select("cidade").distinct().count()}\nValores distintos em estados pre-padronização:{df_customers_silver.select("estado").distinct().count()}')

df_customers_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.dim_vendedores")

In [0]:
# Removendo duplicatas com base na senioridade (timestamp_ingestion mais antigo)
window = Window.partitionBy('id_vendedor').orderBy(col('tempo_ingestao_bronze').desc())
df_sellers_silver = spark.read.table(f"{catalogo}.{silver}.dim_vendedores")

df_dedup = (
    df_sellers_silver
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f'Pre deduplicação senior: {df_sellers_silver.count()}')
print(f'Pós deduplicação senior: {df_dedup.count()}')

df_dedup.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.dim_vendedores")

# Tratamento da Tabela 
## tb_product_category_name_translation -> dim.categoria_produtos_traducao

### Mudanças
- Nomes das colunas alternados para o português

In [0]:
df_bronze_categoria_produto = spark.read.table(f"{catalogo}.bronze.tb_product_category_name_translation")

nomes = {'product_category_name':'nome_produto_pt','product_category_name_english':'nome_produto_en','timestamp_ingestion':'tempo_ingestao_bronze'}

rename_columns(df_bronze_categoria_produto, nomes, 'dim_categoria_produtos_traducao')


# Tratamento da Tabela 
## tb_cotacoes -> dim_cotacao_dolar

### Mudanças
- Criação de um calendário continuo para corrigir erros de nulos

In [0]:
df_cotacao_silver = spark.read.table('medalhao_rocket.bronze.tb_cotacoes')

df_with_date = df_cotacao_silver.withColumn('data', F.to_date(F.col('dataHoraCotacao')))

min_max = df_with_date.agg(F.min('data'), F.max('data')).first()
min_date = min_max[0]
max_date = min_max[1]

df_calendario = spark.createDataFrame([(min_date, max_date)], ['start', 'end']) \
    .withColumn('data', F.explode(F.sequence('start', 'end', F.expr('interval 1 day')))) \
    .select('data')

df_completo = df_calendario.join(df_with_date, on='data', how='left')

window_spec = Window.partitionBy(F.lit(1)).orderBy('data').rowsBetween(Window.unboundedPreceding, 0)

df_cotacao_ajustado_silver = df_completo.withColumn(
    'cotacaoCompra',
    F.last('cotacaoCompra', ignorenulls=True).over(window_spec)
).select('cotacaoCompra', 'data')

df_cotacao_ajustado_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.dim_cotacao_dolar")


# Tabela final Silver

## Adições
- Junção da tabela fat_pedidos com fat_pagamentos_pedidos e dim_cotacao_dolar
- Contém as colunas
  - id_pedido: extraida diretamente da tabela de pedidos, chave primária
  - id_consumidor: extraida da tabela de pedidos
  - status: extraida da tabela de pedidos
  - data_pedido: extraida da tabela de pedidos
  - valor_total_pago_brl: extraida da tabela de pagamentos
  - valor_total_pago_usd: calculada através da cotação do dolar na data de compra baseada no preço de valor_total_pago_brl

In [0]:
from pyspark.sql import functions as F

df_fat_pedidos_silver = spark.read.table(f"{catalogo}.{silver}.fat_pedidos")
df_dim_cotacao_dolar = spark.read.table(f"{catalogo}.{silver}.dim_cotacao_dolar")
df_fat_pagamentos_pedidos = spark.read.table(f"{catalogo}.{silver}.fat_pagamentos_pedidos")

df_final_silver = (
    df_fat_pedidos_silver.alias('ped')
    
    .join(
        df_fat_pagamentos_pedidos.alias('pag'),
        F.col('ped.id_pedido') == F.col('pag.id_pedido'),
        'inner'
    )
    
    .join(
        df_dim_cotacao_dolar.alias('cot'),
        F.to_date(F.col('ped.data_compra')) == F.col('cot.data'),
        'left'
    )
    
    .select(
         F.col('ped.id_pedido').alias('id_pedido'),
         F.col('ped.id_consumidor').alias('id_consumidor'),
         F.col('ped.status_pedido').alias('status'),
         F.col('ped.data_compra').alias('data_pedido'),
         F.col('pag.valor_pagamento').alias('valor_total_pago_brl'),

        F.round(
            F.col('pag.valor_pagamento') / F.col('cot.cotacaoCompra'),
            2
            ).alias('valor_total_pago_usd')
        )
)

df_final_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver}.fat_pedidos_total")

display(df_final_silver)

# Otimização Física Delta

In [0]:
%sql

OPTIMIZE medalhao_rocket.silver.fat_avaliacoes_pedidos
ZORDER BY id_pedido

In [0]:
%sql

OPTIMIZE medalhao_rocket.silver.fat_itens_pedidos
ZORDER BY id_pedido

In [0]:
%sql

OPTIMIZE medalhao_rocket.silver.fat_pagamentos_pedidos
ZORDER BY id_pedido

In [0]:
%sql

OPTIMIZE medalhao_rocket.silver.fat_pedidos
ZORDER BY id_pedido

In [0]:
%sql

OPTIMIZE medalhao_rocket.silver.fat_pedidos_total
ZORDER BY id_pedido, data_pedido
